# 03 — Train Qwen3-8B: Summarization (Nepali)

- **Model:** `unsloth/Qwen3-8B-unsloth-bnb-4bit`
- **Dataset:** xlsum nepali (3,000 examples)
- **Method:** QLoRA r=16 via Unsloth
- **Output:** `iwasbinod/qwen3-8b-nepali-summarization-qlora`

In [ ]:
!pip install unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git -q
!pip install "trl>=0.12.0,<0.16.0" peft accelerate bitsandbytes rouge-score -q

In [ ]:
import sys
sys.path.append('/kaggle/working/final_year_proj_finetuning')
from src.data_utils import load_jsonl
from src.training_utils import load_model, apply_qlora, build_trainer, save_and_push
from src.evaluation import evaluate_summarization
from src.utils import set_seed, hf_login, print_gpu_memory
import os
set_seed(42)
print_gpu_memory()
hf_login(os.environ.get('HF_TOKEN'))

In [ ]:
train_data = load_jsonl('data/processed/summarization_train.jsonl')
val_data   = load_jsonl('data/processed/summarization_val.jsonl')
print(f'Train: {len(train_data)}, Val: {len(val_data)}')
print('\nSample prompt:')
print(train_data[0]['text'][:400])

In [ ]:
model, tokenizer = load_model('qwen3-8b')
model = apply_qlora(model, r=16, lora_alpha=32)
print_gpu_memory()

In [ ]:
trainer = build_trainer(
    model=model, tokenizer=tokenizer, train_data=train_data,
    output_dir='outputs/adapters/qwen3-8b-nepali-summarization-qlora',
    num_epochs=3, batch_size=2, grad_accum=4, lr=2e-4, max_seq_length=512,
)
trainer_stats = trainer.train()
print(f'Training loss: {trainer_stats.training_loss:.4f}')

In [ ]:
results = evaluate_summarization(model, tokenizer, val_data, n_samples=100)
print(f'ROUGE-1={results["rouge1"]}, ROUGE-2={results["rouge2"]}, ROUGE-L={results["rougeL"]}')

In [ ]:
repo_id = save_and_push(model, tokenizer, model_key='qwen3-8b', task='summarization')
print(f'https://huggingface.co/{repo_id}')